# ASL Alphabet Classification Model Training

This notebook handles data loading, preprocessing, and model training

## IMPORT & SETUP

In [1]:
import json
from tensorflow.keras import layers
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Create training and model output directory if it doesn't exist
os.makedirs('../model_training', exist_ok=True)
os.makedirs('../model_outputs', exist_ok=True)

## LOAD DATASET

In [2]:
df = pd.read_csv('../dataset/landmark_dataset.csv')
print(df['label'].value_counts())

X = df.drop('label', axis=1).values  # 63 features
y = df['label'].values

label
A        1000
B        1000
C        1000
D        1000
E        1000
F        1000
G        1000
H        1000
I        1000
K        1000
L        1000
M        1000
N        1000
O        1000
P        1000
Q        1000
R        1000
V        1000
W        1000
Y        1000
space    1000
S         600
T         600
U         600
X         600
del       600
Name: count, dtype: int64


## ENCODE LABELS

In [3]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
class_names = list(encoder.classes_)
print("Classes:", class_names)

Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'del', 'space']


## DATA SPLITTING (80/10/10)

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 19200 | Val: 2400 | Test: 2400


## MODEL ARCHITECTURE

In [5]:
model = tf.keras.Sequential([
    layers.Input(shape=(63,)),
    layers.Dense(128, activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu', kernel_initializer='he_normal'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(len(class_names), activation='softmax'),
])

model.summary(print_fn=lambda x: print(x, flush=True))

Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)          

### COMPILE MODEL

In [6]:
optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    loss= tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

### CALLBACKS SETUP

In [7]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath='../model_outputs/best_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    )
]

### MODEL TRAINING

In [8]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks
)

600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9783 - loss: 0.0717 - val_accuracy: 0.9946 - val_loss: 0.0363 - learning_rate: 7.8125e-06


In [9]:
history.history

{'accuracy': [0.5012500286102295,
  0.8002604246139526,
  0.8578645586967468,
  0.8878124952316284,
  0.9005208611488342,
  0.9155729413032532,
  0.9185937643051147,
  0.9422916769981384,
  0.9458333253860474,
  0.9474999904632568,
  0.9544791579246521,
  0.9509375095367432,
  0.9522916674613953,
  0.9547395706176758,
  0.9580729007720947,
  0.95723956823349,
  0.9579166769981384,
  0.9590104222297668,
  0.9674479365348816,
  0.9707812666893005,
  0.9707291722297668,
  0.973437488079071,
  0.9721354246139526,
  0.9739062786102295,
  0.973229169845581,
  0.9745833277702332,
  0.9750000238418579,
  0.9765625,
  0.9734895825386047,
  0.97635418176651,
  0.9779166579246521,
  0.9758853912353516,
  0.9770312309265137,
  0.9786458611488342,
  0.9785937666893005,
  0.9774479269981384,
  0.9776562452316284,
  0.9783333539962769],
 'loss': [1.6292608976364136,
  0.6174768805503845,
  0.42272433638572693,
  0.3369848132133484,
  0.3048054873943329,
  0.260492742061615,
  0.24470657110214233,
  0

In [10]:
model.evaluate(X_test, y_test)

[0.023002080619335175, 0.996666669845581]

### EXPORT ARTIFACTS

In [11]:
# Saves and Export Datasets
with open('../model_outputs/class_names.json', 'w') as f:
    json.dump(class_names, f)